In [10]:
# import mitsuba
import mitsuba as mi, math

# import drjit
import drjit as dr

import numpy as np
import matplotlib.pyplot as plt

# colocar el valor de spectral
mi.set_variant('cuda_ad_spectral')

from mitsuba import ScalarTransform4f as T
from scipy import constants as const


In [11]:
def gausian(lambdas: np.ndarray, mu: float, sigma: float) -> np.ndarray:
    """
    Calculate the Gaussian function value at x with mean mu and standard deviation sigma.
    Args:
        lambdas (np.ndarray): The input values (wavelengths).
        mu (float): The mean of the Gaussian.
        sigma (float): The standard deviation of the Gaussian.
    """
    return np.exp(-0.5 * ((lambdas - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))


In [12]:
def camera_response_gaussian(wavelength: int, sigma:float, k: float = 3, n_wavelents: int = 5 ):
    """
    Camera response function

    Args:
        wavelength (int): Wavelength in nm.
        sigma (float): Standard deviation of the Gaussian.
        k (float, optional): Number of standard deviations. Default is 3.
        n_wavelents (int, optional): Number of wavelengths. Default is 5.
    """

    # define the wavelength range (n values)
    wavelengths = np.linspace(wavelength - k * sigma, wavelength + k * sigma, n_wavelents)

    # define the Gaussian function
    gaussian_values = gausian(wavelengths, wavelength, sigma)

    # normalize the Gaussian function (0,1)
    gaussian_values = (gaussian_values - np.min(gaussian_values)) / (np.max(gaussian_values) - np.min(gaussian_values))

    return wavelengths, gaussian_values
    


In [13]:
def load_sensor_gaussian(r: float, phi: float, theta: float, wave_lengths: np.ndarray, sigma: float, k: float = 3, n_wavelents: int = 5):
    """
    Load a sensor with Gaussian response.

    Args:
        r (float): Radius of the sensor.
        phi (float): Azimuthal angle in degrees.
        theta (float): Polar angle in degrees.
        wave_lengths (np.ndarray): Wavelengths for the Gaussian response.
        sigma (float): Standard deviation of the Gaussian.
        k (float, optional): Number of standard deviations. Default is 3.
        n_wavelents (int, optional): Number of wavelengths. Default is 5.
    """
    z = r*np.sin(math.radians(theta))
    y = r*np.cos(math.radians(theta))*np.sin(math.radians(phi))
    x = r*np.cos(math.radians(theta))*np.cos(math.radians(phi))
    
    origin = np.array([x, y, z])
    film_dic = {
        'type': 'specfilm',
        'width': 256,
        'height': 256,
        'rfilter': {
            'type': 'tent',
        }
    }

    # create sensor uniform
    distancia = wave_lengths[1] - wave_lengths[0]
    separar = distancia // 2

    for wave_length in wave_lengths:
        
        w_min = int(wave_length - separar)
        w_max = int(wave_length + separar)

        values = '1.0, 1.0'

        film_dic[f'band_{wave_length}'] = {
            'type': 'regular',
            'wavelength_min': w_min,
            'wavelength_max': w_max,
            'values' : values,
        }

    return mi.load_dict({
        'type': 'perspective',
        'fov': 40,
        'to_world': T().look_at(
            origin=origin,
            target=[0, 0, 0],
            up=[0, 0, 1]
        ),
        'sampler': {
            'type': 'independent',
            'sample_count': 1
        },
        'film': film_dic,
        },
    )


In [14]:
# Función para calcular la radiancia del cuerpo negro
def blackbody_radiance_nm(wavelengths_nm, temperature):
    """
    Compute spectral radiance B(λ, T) of a black body
    using scipy constants.

    Args:
        wavelengths_nm: array-like of wavelengths in nanometers (nm).
        temperature:    temperature in Kelvin (K).
    
    Returns:
        numpy array of spectral radiance in W·sr⁻¹·m⁻²·nm⁻¹.
    """
    # Convert wavelengths to meters
    wavelengths_m = np.array(wavelengths_nm, dtype=float) * 1e-9
    
    # Planck's law for spectral radiance per meter: W·sr⁻¹·m⁻²·m⁻¹
    B_m = (2 * const.h * const.c**2) / (wavelengths_m**5) / (
        np.exp(const.h * const.c / (wavelengths_m * const.k * temperature)) - 1
    )
    
    # Convert from per meter to per nanometer: 1 m = 1e9 nm
    B_nm = B_m * 1e-9
    
    return B_nm


In [15]:
def lista_a_string(valores):
    return ", ".join(str(v) for v in valores)


In [16]:
# Configuración inicial
wave_lengths = np.linspace(8000, 14000, 49).astype(int)
sigma = 10
phi = 0
theta = 0
k = 3
n = 15
temperatura = 5000  # temperatura objeto

# Cargar datos de emisividad para la emisión espectral
dataBaseName = np.load('data/matName_FullDatabase.npy', allow_pickle=True).item()["matName"]
dataBaseName = dataBaseName.squeeze() 
dataBaseName = np.hstack(dataBaseName)  # lista de nombres de los materiales

dataBaseLib = np.load('data/matLib_FullDatabase.npy', allow_pickle=True).item()["matLib"] 
dataBaseLib = dataBaseLib[::-1, :]  # Reverso el orden de la base de datos

# Configurar material
material = "stone"  # escoge el material que quieres
indice_material = np.where(dataBaseName == material)[0][0]  # busca el indice del material en la base de datos
firma = dataBaseLib[:, indice_material]  # firma espectral del material

# Calcular la emisión
black_body = blackbody_radiance_nm(wave_lengths, temperatura)  # radiancia del objeto
emision = black_body * firma  # emision del objeto


In [17]:
# Crear un emisor puntual en vez del dragón
point_emitter = {
    'type': 'point',
    'position': [0, 0, 0],  # ubicado en el origen
    'intensity': {
        'type': 'irregular',
        'wavelengths': lista_a_string(wave_lengths),
        'values': lista_a_string(emision),
    }
}


In [18]:
# Crear la escena con el emisor puntual
scene = mi.load_dict({
    'type': 'scene',
    'integrator': {
        'type': 'path',
    },
    'point_light': point_emitter,
})

# Agregar una pequeña esfera para visualizar dónde está el emisor puntual
# Esto es solo para referencia visual, no afecta la emisión del punto de luz
sphere = {
    'type': 'sphere',
    'center': [0, 0, 0],  # Ubicada en el origen (igual que el emisor puntual)
    'radius': 1.0,        # Radio de 1 unidad para mejor visualización
    'bsdf': {
        'type': 'diffuse',
        'reflectance': {
            'type': 'rgb',
            'value': [1.0, 0.8, 0.2]  # Color amarillento para distinguirla mejor
        }
    }
}

# Crear un nuevo diccionario de la escena con todos los elementos
scene_dict = mi.traverse(scene)

# Agregar la esfera como un nuevo elemento en el diccionario
# En vez de modificar el diccionario existente, creamos uno nuevo con la esfera
scene_dict = {
    **scene_dict,  # Mantener todos los elementos existentes
    'sphere': sphere  # Agregar la esfera como un nuevo objeto
}

# Cargar el nuevo diccionario para crear la escena
scene = mi.load_dict(scene_dict)


RuntimeError: ​[xml_v.cpp:170] Missing key 'type' in dictionary: {'point_light.sampling_weight': 1.0, 'point_light.position': [[0, 0, 0]], 'point_light.intensity.wavelengths': [8000, 8125, 8250, .. 43 skipped .., 13750, 13875, 14000], 'point_light.intensity.values': [8.28011, 7.77917, 7.32604, .. 43 skipped .., 0.994094, 0.961821, 0.901232], 'sphere': {'type': 'sphere', 'center': [0, 0, 0], 'radius': 1.0, 'bsdf': {'type': 'diffuse', 'reflectance': {'type': 'rgb', 'value': [1.0, 0.8, 0.2]}}}}

In [ ]:
# Definir los factores de distancia (diferentes a los del dragón)
distancias = [1, 2, 4, 8, 16]  # multiplicadores de distancia: 1x, 2x, 4x, 8x, 16x
base_distance = 35  # distancia base (1x)

# Configuración de muestreo
desarrollo = True  # Cambiar a False para renderizado final de alta calidad
spp = 128 if desarrollo else 1024  # Menos muestras durante desarrollo, más para resultado final

# Lista para almacenar los resultados
imagenes = []
sensores = []
pixeles = []

# Renderizar la escena para cada distancia
for factor in distancias:
    # Calcular la nueva distancia
    r = base_distance * factor
    
    # Crear un sensor a esta distancia
    sensor = load_sensor_gaussian(r, phi, theta, wave_lengths, sigma, k, n)
    sensores.append(sensor)
    
    # Renderizar la escena con este sensor
    print(f"Renderizando imagen a distancia {factor}x ({r} unidades)...")
    imagen = mi.render(scene, sensor=sensor, spp=spp)
    imagenes.append(imagen)
    
    # Extraer valores del pixel central
    x, y = 128, 128  # centro de la imagen
    pixel = imagen[y, x, :].numpy()
    pixeles.append(pixel)


In [ ]:
# Visualizar las imágenes a diferentes distancias
banda = 0  # elegir una banda para visualizar

plt.figure(figsize=(15, 10))
for i, factor in enumerate(distancias):
    plt.subplot(2, 3, i + 1)
    plt.imshow(imagenes[i][:, :, banda], cmap='plasma')
    plt.colorbar()
    plt.title(f'Distancia: {factor}x')
    plt.axis('off')

plt.tight_layout()
plt.savefig('output/point_source_distances_images.png')
plt.show()


In [ ]:
# Comparar las emisiones a diferentes distancias
plt.figure(figsize=(15, 10))

# Primero graficar la emisión original del material
plt.subplot(2, 3, 1)
plt.plot(wave_lengths, emision)
plt.title(f'Emisión original del punto')
plt.xlabel('Longitud de onda (nm)')
plt.ylabel('Emisión (W·sr⁻¹·m⁻²·nm⁻¹)')
plt.grid(True)

# Graficar los valores de píxeles para cada distancia
for i, factor in enumerate(distancias):
    plt.subplot(2, 3, i + 2)
    plt.plot(wave_lengths, pixeles[i])
    plt.title(f'Emisión captada a distancia {factor}x')
    plt.xlabel('Longitud de onda (nm)')
    plt.ylabel('Valor del píxel')
    plt.grid(True)

plt.tight_layout()
plt.savefig('output/point_source_distances_spectra.png')
plt.show()


In [ ]:
# Comparar todas las emisiones en una sola gráfica
plt.figure(figsize=(12, 8))

# Normalizar los valores para una mejor comparación
normalized_pixels = []
for pixel in pixeles:
    if np.max(pixel) > 0:  # evitar división por cero
        normalized_pixel = pixel / np.max(pixel)
    else:
        normalized_pixel = pixel
    normalized_pixels.append(normalized_pixel)

# Graficar todas las emisiones normalizadas
for i, factor in enumerate(distancias):
    plt.plot(wave_lengths, normalized_pixels[i], label=f'Distancia {factor}x')

plt.title('Comparación de emisiones normalizadas a diferentes distancias - Fuente Puntual')
plt.xlabel('Longitud de onda (nm)')
plt.ylabel('Emisión normalizada')
plt.legend()
plt.grid(True)
plt.savefig('output/point_source_distances_comparison.png')
plt.show()


In [ ]:
# Comparación detallada: emisión original vs. medida para cada distancia
plt.figure(figsize=(15, 12))

# Normalizar la emisión original para comparación
emision_norm = emision / np.max(emision)

for i, factor in enumerate(distancias):
    plt.subplot(3, 2, i + 1)
    
    # Graficar la emisión original normalizada
    plt.plot(wave_lengths, emision_norm, 'r--', label='Emisión original')
    
    # Graficar la emisión medida normalizada
    plt.plot(wave_lengths, normalized_pixels[i], 'b-', label=f'Medida a {factor}x')
    
    plt.title(f'Comparación a distancia {factor}x')
    plt.xlabel('Longitud de onda (nm)')
    plt.ylabel('Emisión normalizada')
    plt.grid(True)
    plt.legend()

plt.tight_layout()
plt.savefig('output/point_source_emission_comparison_detail.png')
plt.show()


In [ ]:
# Comparación de la caída de intensidad con la distancia
intensidades = [np.max(pixel) for pixel in pixeles]
intensidad_relativa = [intensidad / intensidades[0] for intensidad in intensidades]

# La ley del cuadrado inverso predice que la intensidad cae con el cuadrado de la distancia
ley_cuadrado_inverso = [1 / (factor**2) for factor in distancias]

plt.figure(figsize=(10, 6))
plt.plot(distancias, intensidad_relativa, 'o-', label='Intensidad medida')
plt.plot(distancias, ley_cuadrado_inverso, 's--', label='Ley del cuadrado inverso (1/r²)')
plt.title('Caída de intensidad con la distancia - Fuente Puntual')
plt.xlabel('Factor de distancia')
plt.ylabel('Intensidad relativa')
plt.xscale('log')
plt.yscale('log')
plt.grid(True)
plt.legend()
plt.savefig('output/point_source_intensity_falloff.png')
plt.show()


In [ ]:
# Análisis del error respecto a la ley del cuadrado inverso
plt.figure(figsize=(10, 6))

# Calcular el error porcentual respecto a la ley del cuadrado inverso
error_porcentual = [(medido / teorico - 1) * 100 
                   for medido, teorico in zip(intensidad_relativa, ley_cuadrado_inverso)]

# Graficar el error porcentual
plt.bar(range(len(distancias)), error_porcentual)
plt.xticks(range(len(distancias)), [f'{d}x' for d in distancias])
plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
plt.title('Error porcentual respecto a la ley del cuadrado inverso')
plt.xlabel('Factor de distancia')
plt.ylabel('Error (%)')
plt.grid(True, axis='y', alpha=0.3)
plt.savefig('output/point_source_inverse_square_error.png')
plt.show()


In [ ]:
# Tabla de resultados
print("\nTabla de resultados - Fuente Puntual")
print("=" * 70)
print(f"{'Distancia':^12} | {'Intensidad':^15} | {'Intensidad rel.':^15} | {'Ley 1/r²':^15} | {'Error %':^10}")
print("-" * 70)
for i, d in enumerate(distancias):
    print(f"{d:^12.0f}x | {intensidades[i]:^15.8f} | {intensidad_relativa[i]:^15.8f} | {ley_cuadrado_inverso[i]:^15.8f} | {error_porcentual[i]:^10.2f}%")
print("=" * 70)
